## Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm


import sys, os
# sys.path.append(os.path.join(os.getcwd(), "../ml-force"))


from ml_force import MorrisLecar, MorrisLecarCurrent, Reservoir
from ml_force.utils import minmax_transform, z_transform
from ml_force.supervisors import LorenzAttractor

from scipy.signal import find_peaks

## Preparations for Lorenz Attractor

In [ ]:
seed = 1
np.random.seed(seed)
torch.cuda.manual_seed(seed)
torch.random.manual_seed(seed)

T = 20000
dt = 1e-1
t = np.arange(0, T, dt)
nt = t.size
# x = VanDerPol(T, dt, mu=1, tau=.02).generate(transient_time=500.0)
x = LorenzAttractor(T, dt, tau=.01).generate(transient_time=1000.0)

x = x.T
# sup = minmax_transform(x, zero_mean=True)
sup = z_transform(x)
print(sup.shape)

In [ ]:
plt.plot(t, sup[:, 1])
plt.show()

# ML Conductance-based Model

In [ ]:
NE = 800
NI = 200
N = NI + NE


# input current for I and E neurons
Ie = 65     # pA
Ii = 65     # pA
current = np.array([Ie] * NE + [Ii] * NI).reshape(N, 1)


Q = 150
lamda = 1.
gbar = 25
sparse = 0.01

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(f"Using Device <{device}> for PyTorch computations...\n")
torch.cuda.random.manual_seed(seed)
res = None
try:
    res = Reservoir(
        n_input=sup.shape[1], n_output=sup.shape[1], w_in_amp=Q, model_cls=MorrisLecar,
        BIAS=current, dt=dt, 
        Ne=NE, Ni=NI, gbar=gbar, device=device,
        )
except Exception as e:
    print(e)
    print(torch.cuda.memory_summary(device=device))

In [ ]:
# colors = plt.pcolormesh(res.w.cpu().numpy(), cmap='viridis')
# plt.colorbar(colors)
# plt.show()
factory_kwargs = res.model.factory_kwargs

print(f"Mean: {res.model.w.type(dtype=torch.float64).mean().item()}, \nSTD: {res.model.w.type(dtype=torch.float64).std().item()}")
print(f"Max: {res.model.w.type(dtype=torch.float64).max().item()}, \nMin: {res.model.w.type(dtype=torch.float64).min().item()}")

In [ ]:
# RLS params
# rls_stop = round(T * .6)
rls_start = 500
test_time = T * .5    # ms
rls_stop = T - test_time
rls_step = 20

memory_length = 5000    # ms
rls_gap = rls_step * dt
memory_coeff = memory_length / rls_gap
ff_coeff = 1 - 1 / memory_coeff

# ff_coeff = 1
ff_coeff

In [ ]:
save_path = os.path.join(os.getcwd(), "results", f"Q_{Q}_gbar_{gbar}_l_{lamda}_Ne_{NE}_Ni_{NI}")
os.makedirs(save_path, exist_ok=True)

def save_tensor(tensor, name):
    file_name = name + ".pt"
    torch.save(tensor, os.path.join(save_path, file_name))
    print(f"Saved {file_name} in {save_path}")
    
def load_tensor(save_path:str, name:str):
    file_name = os.path.join(save_path, name + ".pt")
    tensor = torch.load(os.path.join(save_path, file_name))
    print(f"Loaded {file_name} from {save_path}")
    return tensor

In [ ]:
duration = 500
nt_transient = int(duration / dt)

n_neurons = 10
neurons = np.random.choice(N, n_neurons, replace=False)
print(neurons)

print(f"Transient Period: {duration} ms")
for i in tqdm(range(nt_transient)):
    res.model.forward(0)

sup_tensor = torch.tensor(sup, **factory_kwargs)
nt_train = int((T-test_time) / dt)
s_rec_train = torch.zeros((nt_train, n_neurons), **factory_kwargs)
v_rec_train = torch.zeros((nt_train, n_neurons), **factory_kwargs)
n_rec_train = torch.zeros((nt_train, n_neurons), **factory_kwargs)
dec_rec_train = torch.zeros((nt_train, n_neurons, sup.shape[1]), **factory_kwargs)
xhat_rec_train = torch.zeros((nt_train, sup.shape[1]), **factory_kwargs)

ml = res.model
res.Pinv = torch.eye(res.n_hidden, **factory_kwargs) / lamda

print(f"Training Period: {T-test_time} ms")
for i in tqdm(range(nt_train)):
    x = sup_tensor[i].reshape(-1, 1)       # reshape to (n_input, 1)
    input_ = res.W_in @ x
    ml.forward(input_)
    v_rec_train[i] = ml.mem[neurons].ravel()
    n_rec_train[i] = ml.n[neurons].ravel()
    s_rec_train[i] = ml.s[neurons].ravel()
    x_hat = res.W_out.T @ ml.s                # (n_output, 1)
    xhat_rec_train[i] = x_hat.ravel()
    dec_rec_train[i] = res.W_out[neurons]
    if i <= int(rls_stop / dt) and i % rls_step == 0:
        res._rls(x=x, x_hat=x_hat, state=ml.s, ff_coeff=1.0)

In [ ]:
save_tensor(ml.w, "weights")
save_tensor(s_rec_train, "s_rec_train")
save_tensor(v_rec_train, "v_rec_train")
save_tensor(n_rec_train, "n_rec_train")
save_tensor(xhat_rec_train, "xhat_rec_train")
save_tensor(dec_rec_train, "dec_rec_train")


In [ ]:
t_train = np.arange(nt_train) * dt

fig, ax = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
for i in range(n_neurons):
    ax[0].plot(t_train, minmax_transform(v_rec_train[:, i].cpu().numpy().reshape(-1, 1))+i)
    ax[1].plot(t_train, minmax_transform(n_rec_train[:, i].cpu().numpy().reshape(-1, 1))+i)
    ax[2].plot(t_train, minmax_transform(s_rec_train[:, i].cpu().numpy().reshape(-1, 1))+i)
ax[0].set_title('Voltage')
ax[1].set_title('Potassium Activation')
ax[2].set_title('Synaptic Activation')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5), sharex=True)
ax[0].plot(t_train, dec_rec_train[:, :, 0].cpu().numpy())
ax[0].set_title('Decoding Weights for the x axis')
ax[1].plot(t_train, dec_rec_train[:, :, 1].cpu().numpy())
ax[1].set_title('Decoding Weights for the y axis')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5), sharex=True)
ax[0].plot(t_train, xhat_rec_train[:, 0].cpu().numpy())
ax[0].plot(t_train, sup[:nt_train, 0], '--')
ax[1].plot(t_train, xhat_rec_train[:, 2].cpu().numpy())
ax[1].plot(t_train, sup[:nt_train, 2], '--')
plt.show()

In [ ]:
nt_test = nt - nt_train
s_rec_test = torch.zeros((nt_test, n_neurons), **factory_kwargs)
v_rec_test = torch.zeros((nt_test, n_neurons), **factory_kwargs)
n_rec_test = torch.zeros((nt_test, n_neurons), **factory_kwargs)
x_hat_rec = torch.zeros((nt_test, sup.shape[1]), **factory_kwargs)

fb_matrix = res.W_in @ res.W_out.T          # (n_hidden, n_hidden)  feedback loop matrix
print(f"Testing Period: {test_time} ms")
for i in tqdm(range(nt_test)):
    input_ = fb_matrix @ ml.s
    ml.forward(input_)
    s_rec_test[i] = ml.s[neurons].ravel()
    v_rec_test[i] = ml.mem[neurons].ravel()
    n_rec_test[i] = ml.n[neurons].ravel()
    x_hat = res.W_out.T @ ml.s
    x_hat_rec[i] = x_hat.ravel()

In [ ]:
save_tensor(s_rec_test, "s_rec_test")
save_tensor(v_rec_test, "v_rec_test")
save_tensor(n_rec_test, "n_rec_test")
save_tensor(x_hat_rec, "x_hat_rec")

In [ ]:
res.W_out.abs().max().item()

In [ ]:
s_rec_test = load_tensor(save_path, "s_rec_test")
v_rec_test = load_tensor(save_path, "v_rec_test")
n_rec_test = load_tensor(save_path, "n_rec_test")
x_hat_rec = load_tensor(save_path, "x_hat_rec")

In [ ]:
nt_test = s_rec_test.shape[0]
t_test = np.arange(nt_test) * dt + T - test_time
n_neurons = s_rec_test.shape[1]

fig, ax = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
for i in range(n_neurons):
    ax[0].plot(t_test, minmax_transform(v_rec_test[:, i].cpu().numpy().reshape(-1, 1))+i)
    ax[1].plot(t_test, minmax_transform(n_rec_test[:, i].cpu().numpy().reshape(-1, 1))+i)
    ax[2].plot(t_test, minmax_transform(s_rec_test[:, i].cpu().numpy().reshape(-1, 1))+i)
ax[0].set_title('Voltage')
ax[1].set_title('Potassium Activation')
ax[2].set_title('Synaptic Activation')
plt.show()

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
for i_ax in range(len(ax)):
    ax[i_ax].plot(t_test, x_hat_rec[:, i_ax].cpu().numpy(), label='output')
    ax[i_ax].plot(t_test, sup[-nt_test:, i_ax], '--', label='target')
    ax[i_ax].set_title(f"Test phase, axis: {i_ax}")
    ax[i_ax].legend(loc='upper right')
plt.xlabel('Time [ms]')
plt.suptitle(fr"Ne:{NE}, Ni:{NI}, Q:{Q}, gbar:{gbar}, $\delta$:{lamda}", y=0.95, fontsize=16)
plt.savefig(os.path.join(save_path, f"N_{ml.N}_ff_{ff_coeff}_timeseries.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=sup.shape[1], nrows=2, figsize=(15, 10), sharex=True, sharey=True)
for i in range(sup.shape[1]):
    ax[0, i].plot(sup[-nt_test:, i-1], sup[-nt_test:, i], 'b-', lw=1, label="sup")
    ax[1, i].plot(x_hat_rec[:, i-1].cpu().numpy(), x_hat_rec[:, i].cpu().numpy(), 'r-', lw=1, label="out")
    ax[0, i].set_title(f"Phase Space {i}", fontsize=16)

ax[0, 0].set_ylabel("Target", fontsize=16)
ax[1, 0].set_ylabel("Output", fontsize=16)

# plt.legend()
plt.suptitle(fr"Ne:{NE}, Ni:{NI}, Q:{Q}, $gbar$:{gbar}, $\delta$:{lamda}", y=0.95, fontsize=16)
plt.savefig(os.path.join(save_path, f"N_{ml.N}_ff_{ff_coeff}_phase_space.png"), dpi=300, bbox_inches='tight')
plt.show(fig)

In [ ]:
z = sup[-nt_test:, 2]
z_hat = x_hat_rec[:, 2].cpu().numpy()

dist = 500

peaks = z[find_peaks(z, distance=dist)[0]]
# peaks = peaks[peaks > 0]
peaks_hat = z_hat[find_peaks(z_hat, distance=dist)[0]]
# peaks_hat = peaks_hat[peaks_hat > 0]

ms = 5
alpha = .5
fig = plt.figure(figsize=(8, 8), dpi=300)
ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])
ax.plot(peaks[:-1], peaks[1:], 'bo', ms=ms, label="sup", alpha=alpha)
ax.plot(peaks_hat[:-1], peaks_hat[1:], 'ro', ms=ms, label="out", alpha=alpha)
# plt.xlim(0.5, 3)
# plt.ylim(0.5, 3)
ax.grid(alpha=.5)
ax.legend()
ax.set_title(f"Ie={Ie} pA, Ii={Ii} pA, Q={Q}, gbar={gbar}, l={lamda}", fontsize=16)
plt.savefig(os.path.join(save_path, f"N_{ml.N}_ff_{ff_coeff}_return_map_.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
hists = []
for axis in range(sup.shape[1]):
    sup_hist, bins = np.histogram(sup[-nt_test:, axis], bins=200, density=True)
    output_hist, bins = np.histogram(x_hat_rec[:, axis].cpu().numpy(), bins=bins, density=True)
    hists.append((sup_hist, output_hist, bins))

# KL Divergence
kl_divs = []
for i in range(len(hists)):
    sup_hist, output_hist, _ = hists[i]
    sup_hist = sup_hist / np.sum(sup_hist)  # normalize
    output_hist = output_hist / np.sum(output_hist) # normalize
    # Mask zeros
    sup_hist = np.where(sup_hist == 0, 1e-10, sup_hist)
    output_hist = np.where(output_hist == 0, 1e-10, output_hist)
    # KL Divergence
    kl_div = np.sum(sup_hist * np.log(sup_hist / output_hist))
    print(f"KL Divergence for axis {i}: {kl_div}")
    kl_divs.append(kl_div)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 5), ncols=len(hists))

for axis in range(len(ax)):
    bins = hists[axis][2]
    sup_hist = hists[axis][0]
    output_hist = hists[axis][1]
    # bins = (bins[:-1] + bins[1:]) / 2   # center of the bins
    width = bins[1] - bins[0]
    alpha = .5
    ax[axis].bar(bins[:-1], sup_hist, width=width, label="sup", color='b', alpha=alpha)
    ax[axis].bar(bins[:-1], output_hist, width=width, label="out", color='r', alpha=alpha)
    ax[axis].set_title(f"Histogram for axis {axis}")
    ax[axis].legend(loc='upper right')
plt.suptitle(f"Ie={Ie} pA, Ii={Ii} pA, Q={Q}, gbar={gbar}, $\delta$={lamda}", fontsize=16, y=1.05)
plt.savefig(os.path.join(save_path, f"N_{ml.N}_ff_{ff_coeff}_histogram.png"), dpi=300, bbox_inches='tight')
plt.show()
